# Lab 3 Solution Walkthrough: I2S Audio Transmitter

**Lecture date:** Friday, October 9, 2026

This notebook walks through the Lab 3 reference solution. The goal is not Python programming. The goal is to understand how a precise hardware contract becomes an AI prompt, RTL, a self-checking testbench, and a hardware validation plan.

Students may use chat-based AI, but must write the final design, run the checks, understand the generated code, and explain the hardware measurements.

## Lab 3 timeline

- **October 7:** I2S contract, timing, and Lab 3 assignment.
- **October 9:** Reference solution, golden vectors, testbench, and hardware bring-up walkthrough.
- **October 12-13:** Senior Trip; no class or lab.
- **October 14:** Simulation materials due.
- **October 20:** Lab 3 hardware validation and student teach-back during the Lab 4 meeting.

Required simulation evidence: the corrected design passes and the intentionally broken design fails. Required hardware evidence: BCK, WS, and data are present with the documented timing.

## Hardware contract

The FPGA is the I2S master. The Pico-side receiver samples data on BCK rising edges. The transmitter changes WS and data on generated BCK falling-edge events.

- Master clock: 30.720 MHz
- BCK: toggle every 5 master-clock cycles, giving 3.072 MHz
- Frame: 64 BCK periods, giving 48 kHz
- WS = 0: left slot (I(t) on the DDC SDR); WS = 1: right slot (Q(t) on the DDC SDR)
- Slot: `{sample[23:0], 8'b0}` (assuming you are keeping 24 bits, sent MSB-first)
  - You should think about how many bits you want to keep.  Twenty-four is how many the PCM1808 keeps.
- The first bit must be stable before the rising edge that samples it.

The critical verification question is: does the bit seen by the receiver on each rising edge match the specified 32-bit slot word?

# Generating "Golden Vectors"
Often, you need to generate inputs and outputs, and Python is a good way to do that.  Here is a Python script that generates test vectors so your test bench can test the sawtooth wave.  Here is the sawtooth wave:

In [1]:
from pathlib import Path

FRAME_COUNT = 3
SAW_START = 0x800000
SAW_STEP = 0x010000
SAMPLE_MASK = (1 << 24) - 1

def slot_word(sample):
    return (sample & SAMPLE_MASK) << 8

def frame_lines(frame_index, sample):
    word = slot_word(sample)
    bits = f"{word:032b}"
    return [
        f"frame={frame_index} sample=0x{sample:06x} ws=0 bits={bits}",
        f"frame={frame_index} sample=0x{sample:06x} ws=1 bits={bits}",
    ]

text_lines = [
    "# Each frame contains one WS=0 left slot followed by one WS=1 right slot.",
    "# Each slot is the exact 32 bits sampled on BCK rising edges.",
]
words = []
sample = SAW_START
for frame_index in range(FRAME_COUNT):
    text_lines.extend(frame_lines(frame_index, sample))
    word = slot_word(sample)
    words.extend((f"{word:08x}", f"{word:08x}"))
    sample = (sample + SAW_STEP) & SAMPLE_MASK

print("\n".join(text_lines))
print("\nExpected memory words:")
print("\n".join(words))

# Each frame contains one WS=0 left slot followed by one WS=1 right slot.
# Each slot is the exact 32 bits sampled on BCK rising edges.
frame=0 sample=0x800000 ws=0 bits=10000000000000000000000000000000
frame=0 sample=0x800000 ws=1 bits=10000000000000000000000000000000
frame=1 sample=0x810000 ws=0 bits=10000001000000000000000000000000
frame=1 sample=0x810000 ws=1 bits=10000001000000000000000000000000
frame=2 sample=0x820000 ws=0 bits=10000010000000000000000000000000
frame=2 sample=0x820000 ws=1 bits=10000010000000000000000000000000

Expected memory words:
80000000
80000000
81000000
81000000
82000000
82000000


## What the testbench proves

The testbench acts like the Pico receiver:

1. It observes each `posedge i2s_bck`.
2. It checks the expected WS level for the slot.
3. It shifts in exactly 32 sampled data bits.
4. It compares the captured word against `expected_words.mem`.
5. It stops with `PASS` after six checked slots.

A waveform may look plausible while being shifted by one bit. The negative test is therefore essential: the checker must fail on the intentionally broken right-channel first bit.

In [ ]:
import subprocess
from pathlib import Path

solution = Path.cwd().parent / "ENGR433-Solutions" / "Lab_03"
commands = [
    ["python3", "generate_golden_vectors.py"],
    ["iverilog", "-g2012", "-o", "i2s_pass.vvp", "i2s_tx_demo.sv", "i2s_tb.sv"],
    ["vvp", "i2s_pass.vvp"],
]
for command in commands:
    print("$", " ".join(command))
    result = subprocess.run(command, cwd=solution, text=True, capture_output=True)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {command}")

## Negative test

The deliberately broken transmitter changes the first right-channel bit. Run it with the same testbench:

```sh
cd ENGR433-Solutions/Lab_03
iverilog -g2012 -o i2s_bug.vvp i2s_tx_demo_buggy.sv i2s_tb.sv
vvp i2s_bug.vvp
```

Expected result: `FAIL` on the first corrupted slot, before the timeout. Students should be able to identify which contract rule was violated and why the checker detected it.

## Hardware bring-up and October 20 teach-back

The standalone bring-up top uses the Dev-iCE clock and I2S pins. The solution README contains the synthesis commands. At the Lab 4 meeting, each pair must demonstrate: 

- BCK near 3.072 MHz;
- WS near 48 kHz;
- 32 BCK periods per channel;
- data stable before each sampling rising edge;
- the eight zero-padding bits; and
- a working LED or other documented bring-up indicator.

The teach-back should connect the measured hardware signals to the RTL and the self-checking testbench. Audacity is optional; the required proof is the I2S hardware timing.

## Transition to Lab 4

Lab 3 establishes the pattern used in Lab 4: define a contract, generate bounded RTL with chat AI, make the testbench check behavior rather than appearance, inject a plausible bug, and validate the implementation.

For Lab 4, the new difficulty is that SPI and the DSP logic use different clock domains. The next lecture will introduce the eight-byte frame, malformed-frame tests, shadow registers, and the synchronized commit toggle.